# Run the Fine-Grid Bayesian Experiment

Builds the 49-elevation response operator and compares adaptive and baseline observation policies in paired known-truth simulations.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


## Why this design is defensible


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, re, tempfile, time, warnings, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import nbinom, poisson, chisquare, norm

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


RAW_DIR = PROJECT_ROOT / 'ordem_raw'
REFINEMENT_ZIP = RAW_DIR / 'job-b9824543-c020-42b7-b747-61b298670ce4.zip'
EXPECTED_REFINEMENT_SHA256 = 'eade76d0fcca361e662b8a5da48b67baa809530d3e028b825da445752e75a10a'

OLD_ACTION_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / '33aa70ca_80_plus_baseline'
OLD_RESPONSE_PATH = OLD_ACTION_DIR / 'ordem_action_response_matrix.csv.gz'
CATALOG_PATH = PROJECT_ROOT / 'phase0_outputs' / 'catalog_matching' / 'ca_cfar_1p75_24h_final_audited' / 'final_consolidated_catalog_association.csv'
ASSET_PATH = PROJECT_ROOT / 'phase2_outputs' / 'dynamic_asset_exposure' / 'active_omm_action_audit_v1' / 'active_satellite_shell_exposure.csv'

RADAR_RANGE_MIN_KM = 1026.639
RADAR_RANGE_MAX_KM = 1084.050
PRIMARY_ORDEM_FEATURE = '1mm'
ALTITUDE_EDGES = np.arange(915.0, 1065.0 + 0.1, 15.0)
ALTITUDE_CENTERS = 0.5 * (ALTITUDE_EDGES[:-1] + ALTITUDE_EDGES[1:])
N_LATENT = len(ALTITUDE_CENTERS)

                                                                                                        
N_MAIN_REPLICATES = int(os.environ.get('DEBRIS_FINAL_MAIN_REPS', '200'))
N_SENSITIVITY_REPLICATES = int(os.environ.get('DEBRIS_FINAL_SENS_REPS', '100'))
BUDGET_STEPS = int(os.environ.get('DEBRIS_FINAL_BUDGET', '24'))
SBC_DRAWS = int(os.environ.get('DEBRIS_SBC_DRAWS', '250'))
CHECKPOINT_EVERY = int(os.environ.get('DEBRIS_FINAL_CHECKPOINT_EVERY', '5'))
MASTER_SEED = 20260827
DWELL_HOURS = 1.0
FORCE_RECOMPUTE = False

MAIN_SCENARIOS = (
    'model_matched',
    'localized_anomaly',
    'neighbor_shell_mixing',
    'ordem_response_perturbation',
)
SENSITIVITY_STRENGTHS = (0.5, 2.0)
POLICIES = (
    'best_fixed',
    'random',
    'ordem_flux_fixed',
    'unweighted_aopt',
    'eig',
    'asset_aopt',
)

                                                               
FALSE_EVENTS_UPPER_PER_24H = 0.973
FALSE_RATE_PER_HOUR = FALSE_EVENTS_UPPER_PER_24H / 24.0

print('research root:', PROJECT_ROOT)
print('Main replicates per scenario:', N_MAIN_REPLICATES)
print('Sensitivity replicates per strength:', N_SENSITIVITY_REPLICATES)
print('Observation budget:', BUDGET_STEPS, 'one-hour observations')
print('Policies:', POLICIES)
print('CPU runtime is sufficient. Checkpoints make the full run resumable.')


## 1. Audit and parse the completed ORDEM refinement


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)


def result_directories(root):
    return sorted({path.parent for path in Path(root).rglob('ORDEM.IN')})


def read_ordem_table(path):
    lines = Path(path).read_text(errors='replace').splitlines()
    header_index = next(i for i, line in enumerate(lines) if line.lstrip().startswith('Element'))
    columns = lines[header_index].split()
    rows = []
    for line in lines[header_index + 1:]:
        fields = line.split()
        if len(fields) != len(columns):
            continue
        try:
            rows.append([float(value) for value in fields])
        except ValueError:
            continue
    frame = pd.DataFrame(rows, columns=columns)
    if frame.empty:
        raise ValueError(f'No numeric table parsed from {path}')
    frame['Element'] = frame['Element'].astype(int)
    return frame


def parse_ordem_input(path):
    values = {}
    for line in Path(path).read_text(errors='replace').splitlines():
        if '!' not in line:
            continue
        raw, comment = line.split('!', 1)
        raw, comment = raw.strip(), comment.strip().lower()
        if not raw:
            continue
        if 'year of observation' in comment:
            values['year'] = int(float(raw))
        elif 'sensor latitude' in comment:
            values['latitude_deg'] = float(raw)
        elif comment.startswith('azimuth'):
            values['azimuth_deg'] = float(raw)
        elif comment.startswith('elevation'):
            values['elevation_deg'] = float(raw)
        elif 'file defining all igloo element boundaries (alt)' in comment:
            values['igloo_file'] = raw
    return values


def overlap_fraction(low, high):
    low = np.asarray(low, dtype=float)
    high = np.asarray(high, dtype=float)
    overlap = np.maximum(0.0, np.minimum(high, RADAR_RANGE_MAX_KM) - np.maximum(low, RADAR_RANGE_MIN_KM))
    return np.divide(overlap, high - low, out=np.zeros_like(overlap), where=high > low)


required_inputs = [REFINEMENT_ZIP, OLD_RESPONSE_PATH, CATALOG_PATH, ASSET_PATH]
missing = [str(path) for path in required_inputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing required completed inputs:\n' + '\n'.join(missing))

refinement_hash = sha256_file(REFINEMENT_ZIP)
if refinement_hash != EXPECTED_REFINEMENT_SHA256:
    raise RuntimeError(f'Refinement ZIP checksum mismatch: {refinement_hash}')

work_root = Path(tempfile.mkdtemp(prefix='debris_final_ordem_'))
safe_extract(REFINEMENT_ZIP, work_root)
result_dirs = result_directories(work_root)
if len(result_dirs) != 40:
    raise RuntimeError(f'Expected 40 refinement result folders; found {len(result_dirs)}')

expected_new_elevations = set(np.round(np.arange(65.0, 89.0 + 0.01, 0.5), 6))
expected_new_elevations -= set(np.round(np.arange(65.0, 85.0 + 0.01, 2.5), 6))
required_names = {
    'ORDEM.IN', 'IGLOOFLUX_TEL.OUT', 'IGLOOFLUX_SIGMARAN_TEL.OUT',
    'IGLOOFLUX_SIGMAPOP_TEL.OUT', 'FLUX_TEL.OUT', 'status.txt',
}

inventory_rows = []
parsed_jobs = []
for directory in result_dirs:
    absent = sorted(name for name in required_names if not (directory / name).is_file())
    if absent:
        raise RuntimeError(f'{directory.name} is missing {absent}')
    if 'Complete' not in (directory / 'status.txt').read_text(errors='replace'):
        raise RuntimeError(f'{directory.name} is not marked Complete')
    meta = parse_ordem_input(directory / 'ORDEM.IN')
    if meta != {
        'year': 2018, 'latitude_deg': 69.59, 'azimuth_deg': 90.0,
        'elevation_deg': meta.get('elevation_deg'), 'igloo_file': 'ALT_5_LEO.TIG',
    }:
        raise RuntimeError(f'Unexpected ORDEM configuration in {directory.name}: {meta}')
    flux = read_ordem_table(directory / 'IGLOOFLUX_TEL.OUT')
    sigma_range = read_ordem_table(directory / 'IGLOOFLUX_SIGMARAN_TEL.OUT')
    sigma_population = read_ordem_table(directory / 'IGLOOFLUX_SIGMAPOP_TEL.OUT')
    if len(flux) != 360 or len(sigma_range) != 360 or len(sigma_population) != 360:
        raise RuntimeError(f'Unexpected ORDEM table length in {directory.name}')
    grid_columns = ['Element', 'alt_low', 'alt_high', 'rng_low', 'rng_high']
    if not np.allclose(flux[grid_columns], sigma_range[grid_columns]) or not np.allclose(flux[grid_columns], sigma_population[grid_columns]):
        raise RuntimeError(f'Flux and uncertainty grids differ in {directory.name}')
    inventory_rows.append({**meta, 'job_key': directory.name, 'complete': True})
    parsed_jobs.append((meta, flux, sigma_range, sigma_population))

inventory = pd.DataFrame(inventory_rows).sort_values('elevation_deg').reset_index(drop=True)
actual_new_elevations = set(np.round(inventory.elevation_deg, 6))
if actual_new_elevations != expected_new_elevations or inventory.elevation_deg.duplicated().any():
    raise RuntimeError(
        f'Refinement elevation mismatch. Missing={sorted(expected_new_elevations - actual_new_elevations)}, '
        f'extra={sorted(actual_new_elevations - expected_new_elevations)}'
    )

print('ORDEM REFINEMENT AUDIT PASSED')
print('  Completed jobs:', len(inventory))
print('  Elevation range:', inventory.elevation_deg.min(), 'to', inventory.elevation_deg.max(), 'degrees')
print('  ZIP SHA256:', refinement_hash)


## 2. Build the 49-action response operator and validate its spacing


In [ ]:
FEATURE_COLUMN = f'window_contribution_gt_{PRIMARY_ORDEM_FEATURE}_m2_yr'
FLUX_SUFFIX = '30'                                        


def action_id(elevation):
    return f'az090p0_el{elevation:05.1f}'.replace('.', 'p')


def coarse_response_from_flux(meta, flux):
    flux_columns = [column for column in flux.columns if column.startswith('Flux_') and column.endswith(FLUX_SUFFIX)]
    if len(flux_columns) != 5:
        raise RuntimeError(f"EL={meta['elevation_deg']}: expected five material populations for >1 mm")
    contribution = flux[flux_columns].sum(axis=1).to_numpy() * overlap_fraction(flux.rng_low, flux.rng_high)
    altitude_mid = 0.5 * (flux.alt_low.to_numpy() + flux.alt_high.to_numpy())
    latent_bin = np.searchsorted(ALTITUDE_EDGES, altitude_mid, side='right') - 1
    row = np.zeros(N_LATENT, dtype=float)
    for index in range(N_LATENT):
        row[index] = contribution[latent_bin == index].sum()
    return row


new_rows = []
uncertainty_element_rows = []
response_uncertainty_rows = []
for meta, flux, sigma_range, sigma_population in parsed_jobs:
    elevation = float(meta['elevation_deg'])
    new_rows.append({'elevation_deg': elevation, 'response': coarse_response_from_flux(meta, flux)})

    flux_columns = [column for column in flux.columns if column.startswith('Flux_') and column.endswith(FLUX_SUFFIX)]
    ran_columns = [f"Sig(ran)_{column.split('_', 1)[1]}" for column in flux_columns]
    pop_columns = [f"Sig(pop)_{column.split('_', 1)[1]}" for column in flux_columns]
    overlap = overlap_fraction(flux.rng_low, flux.rng_high)
    summed_flux = flux[flux_columns].sum(axis=1).to_numpy() * overlap
    summed_ran = np.sqrt(np.square(sigma_range[ran_columns].to_numpy()).sum(axis=1)) * overlap
    summed_pop = np.sqrt(np.square(sigma_population[pop_columns].to_numpy()).sum(axis=1)) * overlap
    combined_sigma = np.hypot(summed_ran, summed_pop)
    active = summed_flux > 0
    for i in np.where(active)[0]:
        uncertainty_element_rows.append({
            'elevation_deg': elevation,
            'Element': int(flux.Element.iloc[i]),
            'alt_low': float(flux.alt_low.iloc[i]),
            'alt_high': float(flux.alt_high.iloc[i]),
            'flux_gt_1mm_window_m2_yr': float(summed_flux[i]),
            'sigma_range_m2_yr': float(summed_ran[i]),
            'sigma_population_m2_yr': float(summed_pop[i]),
            'relative_sigma': float(combined_sigma[i] / summed_flux[i]),
        })
    total_flux = float(summed_flux.sum())
    total_sigma = float(np.sqrt(np.square(combined_sigma).sum()))
    response_uncertainty_rows.append({
        'elevation_deg': elevation,
        'window_flux_gt_1mm_m2_yr': total_flux,
        'window_sigma_quadrature_m2_yr': total_sigma,
        'relative_sigma': total_sigma / total_flux,
    })

uncertainty_elements = pd.DataFrame(uncertainty_element_rows)
response_uncertainty = pd.DataFrame(response_uncertainty_rows).sort_values('elevation_deg').reset_index(drop=True)

old_response = pd.read_csv(OLD_RESPONSE_PATH)
required_old_columns = {'azimuth_deg', 'elevation_deg', 'alt_low', 'alt_high', FEATURE_COLUMN}
if not required_old_columns.issubset(old_response.columns):
    raise RuntimeError(f'Old response library is missing: {required_old_columns - set(old_response.columns)}')
old_centerline = old_response.loc[np.isclose(old_response.azimuth_deg, 90.0)].copy()
old_elevations = np.arange(65.0, 85.0 + 0.01, 2.5)
if set(np.round(old_centerline.elevation_deg.unique(), 6)) != set(np.round(old_elevations, 6)):
    raise RuntimeError('The existing azimuth-90° response library does not contain the expected nine elevations.')

old_centerline['altitude_mid_km'] = 0.5 * (old_centerline.alt_low + old_centerline.alt_high)
old_centerline['latent_bin'] = np.searchsorted(ALTITUDE_EDGES, old_centerline.altitude_mid_km, side='right') - 1
old_rows = []
for elevation, group in old_centerline.groupby('elevation_deg'):
    row = np.zeros(N_LATENT, dtype=float)
    for latent_bin, bin_group in group.groupby('latent_bin'):
        if 0 <= int(latent_bin) < N_LATENT:
            row[int(latent_bin)] = bin_group[FEATURE_COLUMN].sum()
    old_rows.append({'elevation_deg': float(elevation), 'response': row})

combined_rows = old_rows + new_rows
combined_rows = sorted(combined_rows, key=lambda item: item['elevation_deg'])
ELEVATIONS = np.asarray([item['elevation_deg'] for item in combined_rows], dtype=float)
H = np.vstack([item['response'] for item in combined_rows])
N_ACTIONS = len(ELEVATIONS)
ACTION_IDS = [action_id(value) for value in ELEVATIONS]

expected_all = np.arange(65.0, 89.0 + 0.01, 0.5)
if N_ACTIONS != 49 or not np.allclose(ELEVATIONS, expected_all):
    raise RuntimeError(f'Expected the exact 49-action 0.5° grid; found {ELEVATIONS.tolist()}')
if len(set(ACTION_IDS)) != 49 or not np.isfinite(H).all() or (H < 0).any() or np.any(H.sum(axis=1) <= 0):
    raise RuntimeError('Invalid fine-grid action-response matrix.')

                                                                                                    
loo_rows = []
for index in range(1, N_ACTIONS - 1):
    predicted = 0.5 * (H[index - 1] + H[index + 1])
    observed = H[index]
    total_relative_error = abs(predicted.sum() - observed.sum()) / observed.sum()
    observed_shape = observed / observed.sum()
    predicted_shape = predicted / predicted.sum()
    loo_rows.append({
        'elevation_deg': ELEVATIONS[index],
        'total_flux_relative_error': float(total_relative_error),
        'normalized_shape_l1_error': float(np.abs(predicted_shape - observed_shape).sum()),
        'normalized_shape_rmse': float(np.sqrt(np.mean(np.square(predicted_shape - observed_shape)))),
    })
loo = pd.DataFrame(loo_rows)

normalized_shapes = H / H.sum(axis=1, keepdims=True)
centered_shapes = normalized_shapes - normalized_shapes.mean(axis=0, keepdims=True)
singular_values = np.linalg.svd(centered_shapes, full_matrices=False, compute_uv=False)
shape_variance = np.square(singular_values)
shape_variance_fraction = shape_variance / shape_variance.sum()
effective_rank = float(np.exp(-np.sum(shape_variance_fraction[shape_variance_fraction > 0] * np.log(shape_variance_fraction[shape_variance_fraction > 0]))))
components_for_99 = int(np.searchsorted(np.cumsum(shape_variance_fraction), 0.99) + 1)

neighbor_cosine = np.sum(normalized_shapes[:-1] * normalized_shapes[1:], axis=1) / (
    np.linalg.norm(normalized_shapes[:-1], axis=1) * np.linalg.norm(normalized_shapes[1:], axis=1)
)
action_audit = {
    'actions': N_ACTIONS,
    'azimuth_deg': 90.0,
    'elevation_min_deg': float(ELEVATIONS.min()),
    'elevation_max_deg': float(ELEVATIONS.max()),
    'spacing_deg': 0.5,
    'effective_shape_rank': effective_rank,
    'components_for_99_percent_shape_variance': components_for_99,
    'neighbor_cosine_similarity_min': float(neighbor_cosine.min()),
    'neighbor_cosine_similarity_median': float(np.median(neighbor_cosine)),
    'loo_total_flux_error_median': float(loo.total_flux_relative_error.median()),
    'loo_total_flux_error_95th': float(loo.total_flux_relative_error.quantile(0.95)),
    'loo_shape_l1_error_median': float(loo.normalized_shape_l1_error.median()),
    'loo_shape_l1_error_95th': float(loo.normalized_shape_l1_error.quantile(0.95)),
    'primary_action_interpretation': '49 exact sampled elevations; interpolation is not used in primary results',
}

print('FINE-GRID ACTION AUDIT PASSED')
print('  Exact elevations:', N_ACTIONS)
print('  Grid:', ELEVATIONS.min(), 'to', ELEVATIONS.max(), 'in 0.5° steps')
print('  Effective normalized-shape rank:', round(effective_rank, 3))
print('  Components explaining 99%:', components_for_99)
print('  Leave-one-out total-flux error, median / 95th:',
      f"{100*action_audit['loo_total_flux_error_median']:.2f}% / {100*action_audit['loo_total_flux_error_95th']:.2f}%")
print('  Leave-one-out shape L1 error, median / 95th:',
      f"{100*action_audit['loo_shape_l1_error_median']:.2f}% / {100*action_audit['loo_shape_l1_error_95th']:.2f}%")


## 3. Real radar counts, conservative dispersion, active-satellite weights, and the robust prior


In [ ]:
catalog = pd.read_csv(CATALOG_PATH)
assets = pd.read_csv(ASSET_PATH)

if len(catalog) != 155:
    raise RuntimeError(f'Expected 155 frozen consolidated events; found {len(catalog)}')
tier_counts = catalog.association_tier.value_counts().to_dict()
if tier_counts != {'unresolved': 106, 'tier_a_strict': 46, 'tier_b_empirical': 3}:
    raise RuntimeError(f'Frozen catalog tiers changed: {tier_counts}')

hourly_counts_series = catalog.groupby('hour_key').size().sort_index()
if len(hourly_counts_series) != 24 or hourly_counts_series.sum() != 155:
    raise RuntimeError('Expected exactly 24 real hourly bins summing to 155 accepted events.')
HOURLY_COUNTS = hourly_counts_series.to_numpy(dtype=float)
HOURLY_LABELS = hourly_counts_series.index.astype(str).to_numpy()
HOURLY_MEAN = float(HOURLY_COUNTS.mean())
HOURLY_VARIANCE = float(HOURLY_COUNTS.var(ddof=1))
POINT_DISPERSION = float(HOURLY_MEAN**2 / (HOURLY_VARIANCE - HOURLY_MEAN)) if HOURLY_VARIANCE > HOURLY_MEAN else 1e6

dispersion_rng = np.random.default_rng(MASTER_SEED)
dispersion_bootstrap = []
for _ in range(50000):
    sample = dispersion_rng.choice(HOURLY_COUNTS, len(HOURLY_COUNTS), replace=True)
    mean = sample.mean()
    variance = sample.var(ddof=1)
    if variance > mean:
        dispersion_bootstrap.append(mean**2 / (variance - mean))
dispersion_bootstrap = np.asarray(dispersion_bootstrap, dtype=float)
INFERENCE_DISPERSION = float(np.quantile(dispersion_bootstrap, 0.05))

SIGNAL_RATE_CENTER_PER_HOUR = max(HOURLY_MEAN - FALSE_RATE_PER_HOUR, 1e-9)
CENTER_ACTION_INDEX = int(np.where(np.isclose(ELEVATIONS, 75.0))[0][0])
CENTER_RESPONSE_TOTAL = float(H[CENTER_ACTION_INDEX].sum())

asset_required = {'altitude_center_km', 'active_asset_exposure_weight'}
if not asset_required.issubset(assets.columns):
    raise RuntimeError(f'Active-satellite input is missing {asset_required - set(assets.columns)}')
asset_lookup = assets.set_index('altitude_center_km')['active_asset_exposure_weight']
if not set(np.round(ALTITUDE_CENTERS, 6)).issubset(set(np.round(asset_lookup.index.to_numpy(dtype=float), 6))):
    raise RuntimeError('Active-satellite shell centers do not align with the Bayesian altitude grid.')
ASSET_WEIGHTS = np.asarray([asset_lookup.loc[center] for center in ALTITUDE_CENTERS], dtype=float)
ASSET_WEIGHTS /= ASSET_WEIGHTS.sum()
EQUAL_WEIGHTS = np.repeat(1.0 / N_LATENT, N_LATENT)

                                                                                         
prior_cv = float(uncertainty_elements.relative_sigma.replace([np.inf, -np.inf], np.nan).dropna().median())
PRIOR_LOG_SD = float(np.sqrt(np.log1p(prior_cv**2)))
RESPONSE_RELATIVE_SD = float(response_uncertainty.relative_sigma.median())
RESPONSE_LOG_SD = float(np.sqrt(np.log1p(RESPONSE_RELATIVE_SD**2)))
if not (0.4 <= PRIOR_LOG_SD <= 1.0):
    raise RuntimeError(f'ORDEM-derived prior log-SD is outside the audited range: {PRIOR_LOG_SD}')

PRIOR_CORRELATION_LENGTH_KM = 30.0
PRIOR_NUGGET_FRACTION = 0.25
distance = np.abs(ALTITUDE_CENTERS[:, None] - ALTITUDE_CENTERS[None, :])
scaled_distance = np.sqrt(3.0) * distance / PRIOR_CORRELATION_LENGTH_KM
matern32 = (1.0 + scaled_distance) * np.exp(-scaled_distance)
K_PRIOR = PRIOR_LOG_SD**2 * (
    (1.0 - PRIOR_NUGGET_FRACTION) * matern32 + PRIOR_NUGGET_FRACTION * np.eye(N_LATENT)
)
K_PRIOR += np.eye(N_LATENT) * 1e-8
K_INV = np.linalg.inv(K_PRIOR)
L_PRIOR = np.linalg.cholesky(K_PRIOR)
MU_PRIOR = np.zeros(N_LATENT)
PRIOR_LOGDET = float(np.linalg.slogdet(K_PRIOR)[1])

print('REAL-CALIBRATED MODEL INPUTS READY')
print('  Accepted-event mean / variance:', round(HOURLY_MEAN, 3), '/', round(HOURLY_VARIANCE, 3))
print('  NB point dispersion:', round(POINT_DISPERSION, 3))
print('  Conservative bootstrap 5th-percentile dispersion:', round(INFERENCE_DISPERSION, 3))
print('  ORDEM uncertainty element observations:', len(uncertainty_elements))
print('  Prior log-SD:', round(PRIOR_LOG_SD, 3))
print('  Median action-level ORDEM response relative SD:', round(RESPONSE_RELATIVE_SD, 3))
print('  Active-satellite shell weights:', np.round(ASSET_WEIGHTS, 4).tolist())


## 4. Blocked held-out prediction of real hourly counts


In [ ]:
def nb_logpmf(y, mean, dispersion):
    probability = dispersion / (dispersion + mean)
    return float(nbinom.logpmf(y, dispersion, probability))


def nb_interval(mean, dispersion, level):
    probability = dispersion / (dispersion + mean)
    alpha = 1.0 - level
    return float(nbinom.ppf(alpha / 2.0, dispersion, probability)), float(nbinom.ppf(1.0 - alpha / 2.0, dispersion, probability))


cv_rows = []
fold_size = 4
for fold in range(6):
    test_indices = np.arange(fold * fold_size, (fold + 1) * fold_size)
    train_indices = np.setdiff1d(np.arange(24), test_indices)
    train = HOURLY_COUNTS[train_indices]
    train_mean = float(train.mean())
    train_variance = float(train.var(ddof=1))
    train_dispersion = float(train_mean**2 / (train_variance - train_mean)) if train_variance > train_mean else 1e6
    for index in test_indices:
        observed = int(HOURLY_COUNTS[index])
        nb80 = nb_interval(train_mean, train_dispersion, 0.80)
        nb95 = nb_interval(train_mean, train_dispersion, 0.95)
        pois80 = (float(poisson.ppf(0.10, train_mean)), float(poisson.ppf(0.90, train_mean)))
        pois95 = (float(poisson.ppf(0.025, train_mean)), float(poisson.ppf(0.975, train_mean)))
        cv_rows.append({
            'fold': fold + 1,
            'hour_key': HOURLY_LABELS[index],
            'observed_events': observed,
            'training_mean': train_mean,
            'training_nb_dispersion': train_dispersion,
            'nb_log_score': nb_logpmf(observed, train_mean, train_dispersion),
            'poisson_log_score': float(poisson.logpmf(observed, train_mean)),
            'nb_80_covered': nb80[0] <= observed <= nb80[1],
            'nb_95_covered': nb95[0] <= observed <= nb95[1],
            'poisson_80_covered': pois80[0] <= observed <= pois80[1],
            'poisson_95_covered': pois95[0] <= observed <= pois95[1],
            'nb_80_low': nb80[0], 'nb_80_high': nb80[1],
            'nb_95_low': nb95[0], 'nb_95_high': nb95[1],
        })

count_cv = pd.DataFrame(cv_rows)
count_cv_summary = {
    'blocked_folds': 6,
    'test_hours_per_fold': 4,
    'nb_mean_log_score': float(count_cv.nb_log_score.mean()),
    'poisson_mean_log_score': float(count_cv.poisson_log_score.mean()),
    'nb_minus_poisson_mean_log_score': float((count_cv.nb_log_score - count_cv.poisson_log_score).mean()),
    'nb_80_coverage': float(count_cv.nb_80_covered.mean()),
    'nb_95_coverage': float(count_cv.nb_95_covered.mean()),
    'poisson_80_coverage': float(count_cv.poisson_80_covered.mean()),
    'poisson_95_coverage': float(count_cv.poisson_95_covered.mean()),
    'interpretation': 'Validates hourly count variability only; it does not validate adaptive steering or individual event truth.',
}

print('BLOCKED REAL-COUNT CROSS-VALIDATION COMPLETE')
display(pd.DataFrame([count_cv_summary]))


## 5. Posterior approximation and predeclared policies


In [ ]:
def stable_seed(*parts):
    payload = '|'.join(map(str, parts)).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], 'little') % (2**32 - 1)


def signal_scale(strength):
    return strength * SIGNAL_RATE_CENTER_PER_HOUR * DWELL_HOURS / CENTER_RESPONSE_TOTAL


def expected_counts_all_actions(x, strength, response_matrix=None):
    operator = H if response_matrix is None else response_matrix
    signal = signal_scale(strength) * (operator @ np.exp(x))
    return FALSE_RATE_PER_HOUR * DWELL_HOURS + signal


def negative_binomial_draw(rng, mean, dispersion):
    mean = np.asarray(mean, dtype=float)
    probability = dispersion / (dispersion + mean)
    return rng.negative_binomial(dispersion, probability)


def posterior_map_cov(history_actions, history_counts, strength, start=None):
    actions = np.asarray(history_actions, dtype=int)
    counts = np.asarray(history_counts, dtype=float)
    if len(actions) == 0:
        return MU_PRIOR.copy(), K_PRIOR.copy(), True, 0

    scale = signal_scale(strength)
    observed_operator = H[actions]
    k = INFERENCE_DISPERSION

    def objective_gradient(z):
        x = MU_PRIOR + L_PRIOR @ z
        exp_x = np.exp(x)
        signal_components = scale * observed_operator * exp_x[None, :]
        means = FALSE_RATE_PER_HOUR * DWELL_HOURS + signal_components.sum(axis=1)
        means = np.maximum(means, 1e-12)
        log_likelihood = (
            gammaln(counts + k) - gammaln(k) - gammaln(counts + 1.0)
            + k * np.log(k) + counts * np.log(means)
            - (counts + k) * np.log(means + k)
        )
        objective = 0.5 * z @ z - log_likelihood.sum()
        derivative_nll_mean = k * (means - counts) / (means * (means + k))
        gradient_x = (derivative_nll_mean[:, None] * signal_components).sum(axis=0)
        gradient_z = z + L_PRIOR.T @ gradient_x
        return float(objective), gradient_z

    initial_x = MU_PRIOR.copy() if start is None else np.asarray(start, dtype=float)
    initial_z = np.linalg.solve(L_PRIOR, initial_x - MU_PRIOR)
    result = minimize(
        objective_gradient,
        initial_z,
        method='L-BFGS-B',
        jac=True,
        bounds=[(-6.0, 6.0)] * N_LATENT,
        options={'maxiter': 100, 'maxls': 35, 'ftol': 1e-10, 'gtol': 1e-6},
    )
    mu = MU_PRIOR + L_PRIOR @ np.asarray(result.x, dtype=float)

    exp_mu = np.exp(mu)
    signal_components = scale * observed_operator * exp_mu[None, :]
    means = FALSE_RATE_PER_HOUR * DWELL_HOURS + signal_components.sum(axis=1)
    observation_variance = means + means**2 / k
    whitened_gradients = signal_components @ L_PRIOR
    whitened_precision = np.eye(N_LATENT) + whitened_gradients.T @ (
        whitened_gradients / observation_variance[:, None]
    )
    whitened_precision = 0.5 * (whitened_precision + whitened_precision.T)
    covariance = L_PRIOR @ np.linalg.inv(whitened_precision) @ L_PRIOR.T
    covariance = 0.5 * (covariance + covariance.T)
    gradient_norm = float(np.linalg.norm(result.jac, ord=np.inf))
    converged = bool(result.success or (np.isfinite(result.fun) and gradient_norm < 1e-4))
    return mu, covariance, converged, int(result.nit)


def theta_moments(mu, covariance):
    diagonal = np.maximum(np.diag(covariance), 0.0)
    mean = np.exp(mu + 0.5 * diagonal)
    variance = (np.exp(diagonal) - 1.0) * np.exp(2.0 * mu + diagonal)
    return mean, variance


def interval_score(truth, low, high, alpha=0.05):
    score = high - low
    score += (2.0 / alpha) * (low - truth) * (truth < low)
    score += (2.0 / alpha) * (truth - high) * (truth > high)
    return score


def posterior_metrics(mu, covariance, truth_x):
    diagonal = np.maximum(np.diag(covariance), 0.0)
    sd = np.sqrt(diagonal)
    posterior_theta_mean, posterior_theta_variance = theta_moments(mu, covariance)
    truth_theta = np.exp(truth_x)
    low95, high95 = np.exp(mu - 1.96 * sd), np.exp(mu + 1.96 * sd)
    low80, high80 = np.exp(mu - norm.ppf(0.90) * sd), np.exp(mu + norm.ppf(0.90) * sd)
    low50, high50 = np.exp(mu - norm.ppf(0.75) * sd), np.exp(mu + norm.ppf(0.75) * sd)
    squared_error = np.square(posterior_theta_mean - truth_theta)
    sign, logdet = np.linalg.slogdet(covariance)
    return {
        'asset_weighted_posterior_variance': float(np.sum(ASSET_WEIGHTS * posterior_theta_variance)),
        'equal_weighted_posterior_variance': float(np.sum(EQUAL_WEIGHTS * posterior_theta_variance)),
        'asset_weighted_rmse': float(np.sqrt(np.sum(ASSET_WEIGHTS * squared_error))),
        'equal_weighted_rmse': float(np.sqrt(np.sum(EQUAL_WEIGHTS * squared_error))),
        'asset_weighted_interval_score_95': float(np.sum(ASSET_WEIGHTS * interval_score(truth_theta, low95, high95))),
        'cellwise_50_coverage': float(np.mean((truth_theta >= low50) & (truth_theta <= high50))),
        'cellwise_80_coverage': float(np.mean((truth_theta >= low80) & (truth_theta <= high80))),
        'cellwise_95_coverage': float(np.mean((truth_theta >= low95) & (truth_theta <= high95))),
        'asset_weighted_95_coverage': float(np.sum(ASSET_WEIGHTS * ((truth_theta >= low95) & (truth_theta <= high95)))),
        'information_gain_nats': float(0.5 * (PRIOR_LOGDET - logdet)) if sign > 0 else np.nan,
        'posterior_volume_fraction': float(np.exp(0.5 * (logdet - PRIOR_LOGDET) / N_LATENT)) if sign > 0 else np.nan,
    }


def fisher_terms(mu, covariance, strength):
    theta_local = np.exp(mu + 0.5 * np.maximum(np.diag(covariance), 0.0))
    gradients = signal_scale(strength) * H * theta_local[None, :]
    means = FALSE_RATE_PER_HOUR * DWELL_HOURS + gradients.sum(axis=1)
    observation_variance = means + means**2 / INFERENCE_DISPERSION
    projected = gradients @ covariance
    projected_variance = np.einsum('ai,ai->a', projected, gradients)
    denominator = observation_variance + projected_variance
    return gradients, means, observation_variance, projected, projected_variance, denominator


def aopt_scores(mu, covariance, strength, weights):
    _, _, _, projected, _, denominator = fisher_terms(mu, covariance, strength)
    diagonal = np.maximum(np.diag(covariance), 0.0)
    variance_before = (np.exp(diagonal) - 1.0) * np.exp(2.0 * mu + diagonal)
    updated_diagonal = diagonal[None, :] - np.square(projected) / denominator[:, None]
    updated_diagonal = np.maximum(updated_diagonal, 0.0)
    variance_after = (np.exp(updated_diagonal) - 1.0) * np.exp(2.0 * mu[None, :] + updated_diagonal)
    return np.sum(weights[None, :] * (variance_before[None, :] - variance_after), axis=1)


def eig_scores(mu, covariance, strength):
    _, _, observation_variance, _, projected_variance, _ = fisher_terms(mu, covariance, strength)
    return 0.5 * np.log1p(np.maximum(projected_variance / observation_variance, 0.0))


def covariance_rank_one_update(mu, covariance, action, strength):
    gradients, _, observation_variance, projected, projected_variance, _ = fisher_terms(mu, covariance, strength)
    denominator = observation_variance[action] + projected_variance[action]
    vector = projected[action]
    updated = covariance - np.outer(vector, vector) / denominator
    return 0.5 * (updated + updated.T)


                                                                                   
                                                                                    
                                                                        
fixed_design_rows = []
for action in range(N_ACTIONS):
    design_covariance = K_PRIOR.copy()
    for _ in range(BUDGET_STEPS):
        design_covariance = covariance_rank_one_update(MU_PRIOR, design_covariance, action, 1.0)
    _, design_variance = theta_moments(MU_PRIOR, design_covariance)
    fixed_design_rows.append({
        'action_index': action,
        'elevation_deg': ELEVATIONS[action],
        'predicted_final_asset_weighted_variance': float(np.sum(ASSET_WEIGHTS * design_variance)),
    })
fixed_design = pd.DataFrame(fixed_design_rows)
BEST_FIXED_ACTION = int(fixed_design.predicted_final_asset_weighted_variance.idxmin())
ORDEM_FLUX_ACTION = int(np.argmax(H.sum(axis=1)))


def choose_action(policy, mu, covariance, strength, rng):
    if policy == 'best_fixed':
        return BEST_FIXED_ACTION, np.nan
    if policy == 'random':
        return int(rng.integers(N_ACTIONS)), np.nan
    if policy == 'ordem_flux_fixed':
        return ORDEM_FLUX_ACTION, np.nan
    if policy == 'unweighted_aopt':
        scores = aopt_scores(mu, covariance, strength, EQUAL_WEIGHTS)
        return int(np.argmax(scores)), float(scores.max())
    if policy == 'eig':
        scores = eig_scores(mu, covariance, strength)
        return int(np.argmax(scores)), float(scores.max())
    if policy == 'asset_aopt':
        scores = aopt_scores(mu, covariance, strength, ASSET_WEIGHTS)
        return int(np.argmax(scores)), float(scores.max())
    raise ValueError(policy)


print('POLICIES PREDECLARED')
print('  Strong best-fixed elevation:', ELEVATIONS[BEST_FIXED_ACTION], 'degrees')
print('  ORDEM-flux-only fixed elevation:', ELEVATIONS[ORDEM_FLUX_ACTION], 'degrees')
print('  Proposed policy: asset_aopt')


## 6. Hidden test worlds


In [ ]:
def mixing_matrix(fraction=0.15):
    matrix = np.zeros((N_LATENT, N_LATENT), dtype=float)
    for source in range(N_LATENT):
        neighbors = []
        if source > 0:
            neighbors.append(source - 1)
        if source < N_LATENT - 1:
            neighbors.append(source + 1)
        matrix[source, source] = 1.0 - fraction
        for destination in neighbors:
            matrix[destination, source] += fraction / len(neighbors)
    return matrix


SHELL_MIXING_MATRIX = mixing_matrix(0.15)


def truth_for_replicate(scenario, replicate, experiment_block, strength):
    rng = np.random.default_rng(stable_seed(MASTER_SEED, 'truth', experiment_block, scenario, strength, replicate))
    base_x = rng.multivariate_normal(MU_PRIOR, K_PRIOR)
    truth_x = base_x.copy()
    truth_operator = H.copy()
    scenario_metadata = {'anomaly_center_km': np.nan, 'response_perturb_log_sd': 0.0, 'mixing_fraction': 0.0}

    if scenario == 'localized_anomaly':
        center_index = int(rng.integers(N_LATENT))
        width_km = float(rng.uniform(8.0, 15.0))
        amplitude = float(rng.uniform(np.log(2.0), np.log(3.0)))
        bump = amplitude * np.exp(-0.5 * np.square((ALTITUDE_CENTERS - ALTITUDE_CENTERS[center_index]) / width_km))
        truth_x = base_x + bump
        scenario_metadata['anomaly_center_km'] = float(ALTITUDE_CENTERS[center_index])
    elif scenario == 'neighbor_shell_mixing':
        truth_theta = SHELL_MIXING_MATRIX @ np.exp(base_x)
        truth_x = np.log(np.maximum(truth_theta, 1e-12))
        scenario_metadata['mixing_fraction'] = 0.15
    elif scenario == 'ordem_response_perturbation':
        raw = rng.normal(size=H.shape)
        smooth = gaussian_filter(raw, sigma=(2.0, 1.0), mode='reflect')
        smooth = (smooth - smooth.mean()) / max(smooth.std(ddof=1), 1e-12)
        multiplier = np.exp(RESPONSE_LOG_SD * smooth - 0.5 * RESPONSE_LOG_SD**2)
        truth_operator = H * multiplier
        scenario_metadata['response_perturb_log_sd'] = RESPONSE_LOG_SD
    elif scenario != 'model_matched':
        raise ValueError(scenario)

    return np.clip(truth_x, -3.0, 3.0), truth_operator, scenario_metadata


def observation_streams(truth_x, truth_operator, strength, replicate, experiment_block, scenario):
    means = expected_counts_all_actions(truth_x, strength, response_matrix=truth_operator)
    rng = np.random.default_rng(stable_seed(MASTER_SEED, 'counts', experiment_block, scenario, strength, replicate))
    streams = np.empty((N_ACTIONS, BUDGET_STEPS), dtype=int)
    for action in range(N_ACTIONS):
        streams[action] = negative_binomial_draw(
            rng, np.repeat(means[action], BUDGET_STEPS), INFERENCE_DISPERSION
        )
    return streams, means


def run_policy(policy, truth_x, streams, strength, scenario, replicate, experiment_block, keep_final=False):
    policy_rng = np.random.default_rng(stable_seed(MASTER_SEED, 'policy', policy, experiment_block, scenario, strength, replicate))
    history_actions, history_counts = [], []
    visits = np.zeros(N_ACTIONS, dtype=int)
    mu, covariance = MU_PRIOR.copy(), K_PRIOR.copy()
    initial = posterior_metrics(mu, covariance, truth_x)
    initial_asset_variance = initial['asset_weighted_posterior_variance']
    initial_equal_variance = initial['equal_weighted_posterior_variance']
    step_rows, action_rows = [], []

    initial.update({
        'experiment_block': experiment_block, 'scenario': scenario, 'strength': strength,
        'replicate': replicate, 'policy': policy, 'step': 0, 'cumulative_events': 0,
        'asset_uncertainty_fraction': 1.0, 'equal_uncertainty_fraction': 1.0,
        'optimizer_success': True, 'optimizer_iterations': 0,
    })
    step_rows.append(initial)

    for step in range(1, BUDGET_STEPS + 1):
        selected, decision_score = choose_action(policy, mu, covariance, strength, policy_rng)
        visit_index = int(visits[selected])
        observed = int(streams[selected, visit_index])
        visits[selected] += 1
        posterior_theta_mean, _ = theta_moments(mu, covariance)
        predicted_before = float(
            FALSE_RATE_PER_HOUR * DWELL_HOURS + signal_scale(strength) * H[selected] @ posterior_theta_mean
        )
        history_actions.append(selected)
        history_counts.append(observed)
        mu, covariance, optimizer_success, optimizer_iterations = posterior_map_cov(
            history_actions, history_counts, strength, start=mu
        )
        metrics = posterior_metrics(mu, covariance, truth_x)
        metrics.update({
            'experiment_block': experiment_block, 'scenario': scenario, 'strength': strength,
            'replicate': replicate, 'policy': policy, 'step': step,
            'cumulative_events': int(sum(history_counts)),
            'asset_uncertainty_fraction': metrics['asset_weighted_posterior_variance'] / initial_asset_variance,
            'equal_uncertainty_fraction': metrics['equal_weighted_posterior_variance'] / initial_equal_variance,
            'optimizer_success': optimizer_success, 'optimizer_iterations': optimizer_iterations,
        })
        step_rows.append(metrics)
        action_rows.append({
            'experiment_block': experiment_block, 'scenario': scenario, 'strength': strength,
            'replicate': replicate, 'policy': policy, 'step': step,
            'action_index': selected, 'action_id': ACTION_IDS[selected],
            'azimuth_deg': 90.0, 'elevation_deg': ELEVATIONS[selected],
            'visit_number': visit_index + 1, 'observed_events': observed,
            'predicted_events_before_update': predicted_before,
            'decision_score': decision_score,
        })
    final_state = {'mu': mu, 'covariance': covariance, 'visits': visits} if keep_final else None
    return step_rows, action_rows, final_state


## 7. Run the paired, checkpointed experiment


In [ ]:
experiment_conditions = []
for scenario in MAIN_SCENARIOS:
    experiment_conditions.append({
        'experiment_block': 'main', 'scenario': scenario, 'strength': 1.0,
        'n_replicates': N_MAIN_REPLICATES,
    })
for strength in SENSITIVITY_STRENGTHS:
    experiment_conditions.append({
        'experiment_block': 'sensitivity', 'scenario': 'model_matched',
        'strength': strength, 'n_replicates': N_SENSITIVITY_REPLICATES,
    })

input_hashes = {
    'refinement_zip_sha256': refinement_hash,
    'old_action_response_sha256': sha256_file(OLD_RESPONSE_PATH),
    'catalog_sha256': sha256_file(CATALOG_PATH),
    'active_satellite_exposure_sha256': sha256_file(ASSET_PATH),
}
experiment_config = {
    'model_version': 'debris_final_fine_grid_nb_laplace_aopt_v1',
    'primary_endpoint': 'final active-satellite-exposure-weighted posterior variance fraction',
    'primary_policy': 'asset_aopt',
    'actions': 49,
    'action_elevations_deg': ELEVATIONS.tolist(),
    'action_azimuth_deg': 90.0,
    'action_interpolation_used_in_primary': False,
    'budget_steps': BUDGET_STEPS,
    'dwell_hours': DWELL_HOURS,
    'main_replicates_per_scenario': N_MAIN_REPLICATES,
    'sensitivity_replicates_per_strength': N_SENSITIVITY_REPLICATES,
    'conditions': experiment_conditions,
    'policies': list(POLICIES),
    'best_fixed_elevation_deg': float(ELEVATIONS[BEST_FIXED_ACTION]),
    'ordem_flux_fixed_elevation_deg': float(ELEVATIONS[ORDEM_FLUX_ACTION]),
    'altitude_edges_km': ALTITUDE_EDGES.tolist(),
    'prior_log_sd': PRIOR_LOG_SD,
    'prior_correlation_length_km': PRIOR_CORRELATION_LENGTH_KM,
    'prior_nugget_fraction': PRIOR_NUGGET_FRACTION,
    'negative_binomial_point_dispersion': POINT_DISPERSION,
    'negative_binomial_inference_dispersion': INFERENCE_DISPERSION,
    'false_rate_per_hour': FALSE_RATE_PER_HOUR,
    'signal_rate_center_per_hour': SIGNAL_RATE_CENTER_PER_HOUR,
    'response_perturbation_log_sd': RESPONSE_LOG_SD,
    'shell_mixing_fraction': 0.15,
    'master_seed': MASTER_SEED,
    'sbc_draws': SBC_DRAWS,
    'input_hashes': input_hashes,
}
config_hash = hashlib.sha256(json.dumps(experiment_config, sort_keys=True).encode()).hexdigest()[:12]
OUTPUT_DIR = PROJECT_ROOT / 'phase3_outputs' / 'final_bayesian_policy' / f'fine_grid_{config_hash}'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'final_experiment_config.json').write_text(json.dumps(experiment_config, indent=2))

all_step_frames, all_action_frames, all_sbc_frames = [], [], []
experiment_started = time.time()
total_replicate_equivalents = sum(condition['n_replicates'] for condition in experiment_conditions)
completed_before = 0

for condition_number, condition in enumerate(experiment_conditions, start=1):
    experiment_block = condition['experiment_block']
    scenario = condition['scenario']
    strength = float(condition['strength'])
    n_replicates = int(condition['n_replicates'])
    token = f"{experiment_block}_{scenario}_s{strength:g}".replace('.', 'p')
    steps_path = CHECKPOINT_DIR / f'steps_{token}.csv.gz'
    actions_path = CHECKPOINT_DIR / f'actions_{token}.csv.gz'
    sbc_path = CHECKPOINT_DIR / f'sbc_{token}.csv.gz'

    if FORCE_RECOMPUTE:
        for path in (steps_path, actions_path, sbc_path):
            if path.exists():
                path.unlink()

    condition_steps = pd.read_csv(steps_path) if steps_path.exists() else pd.DataFrame()
    condition_actions = pd.read_csv(actions_path) if actions_path.exists() else pd.DataFrame()
    condition_sbc = pd.read_csv(sbc_path) if sbc_path.exists() else pd.DataFrame()
    completed_replicates = set(condition_steps.replicate.unique().astype(int)) if len(condition_steps) else set()

    print(f'\nCondition {condition_number}/{len(experiment_conditions)}: {experiment_block} / {scenario} / {strength:g}×')
    print('  Completed checkpoint replicates:', len(completed_replicates), '/', n_replicates)
    pending_steps, pending_actions, pending_sbc = [], [], []

    for replicate in range(n_replicates):
        if replicate in completed_replicates:
            continue
        replicate_started = time.time()
        truth_x, truth_operator, scenario_metadata = truth_for_replicate(
            scenario, replicate, experiment_block, strength
        )
        streams, truth_means = observation_streams(
            truth_x, truth_operator, strength, replicate, experiment_block, scenario
        )
        for policy in POLICIES:
            keep_final = bool(
                policy == 'asset_aopt' and experiment_block == 'main'
                and scenario == 'model_matched' and np.isclose(strength, 1.0)
            )
            step_rows, action_rows, final_state = run_policy(
                policy, truth_x, streams, strength, scenario, replicate, experiment_block,
                keep_final=keep_final,
            )
            for row in step_rows:
                row.update(scenario_metadata)
            pending_steps.extend(step_rows)
            pending_actions.extend(action_rows)

            if keep_final:
                mu = final_state['mu']
                sd = np.sqrt(np.maximum(np.diag(final_state['covariance']), 0.0))
                for latent_index in range(N_LATENT):
                    rank_rng = np.random.default_rng(stable_seed(
                        MASTER_SEED, 'sbc_rank', replicate, latent_index, config_hash
                    ))
                    draws = rank_rng.normal(mu[latent_index], sd[latent_index], size=SBC_DRAWS)
                    pending_sbc.append({
                        'replicate': replicate,
                        'latent_index': latent_index,
                        'altitude_center_km': ALTITUDE_CENTERS[latent_index],
                        'truth_log_multiplier': truth_x[latent_index],
                        'posterior_log_mean': mu[latent_index],
                        'posterior_log_sd': sd[latent_index],
                        'rank': int(np.sum(draws < truth_x[latent_index])),
                        'draws': SBC_DRAWS,
                    })

        completed_replicates.add(replicate)
        elapsed_replicate = time.time() - replicate_started
        elapsed_total = time.time() - experiment_started
        done_equivalents = completed_before + len(completed_replicates)
        eta_minutes = elapsed_total / max(done_equivalents, 1) * (total_replicate_equivalents - done_equivalents) / 60.0
        print(f'  Replicate {replicate + 1:03d}/{n_replicates} in {elapsed_replicate:5.1f}s; ETA {eta_minutes:6.1f} min')

        if len(completed_replicates) % CHECKPOINT_EVERY == 0 or len(completed_replicates) == n_replicates:
            condition_steps = pd.concat([condition_steps, pd.DataFrame(pending_steps)], ignore_index=True)
            condition_actions = pd.concat([condition_actions, pd.DataFrame(pending_actions)], ignore_index=True)
            if pending_sbc:
                condition_sbc = pd.concat([condition_sbc, pd.DataFrame(pending_sbc)], ignore_index=True)
            condition_steps.to_csv(steps_path, index=False, compression='gzip')
            condition_actions.to_csv(actions_path, index=False, compression='gzip')
            if len(condition_sbc):
                condition_sbc.to_csv(sbc_path, index=False, compression='gzip')
            pending_steps, pending_actions, pending_sbc = [], [], []
            print('    checkpoint saved')

    if pending_steps:
        condition_steps = pd.concat([condition_steps, pd.DataFrame(pending_steps)], ignore_index=True)
        condition_actions = pd.concat([condition_actions, pd.DataFrame(pending_actions)], ignore_index=True)
        if pending_sbc:
            condition_sbc = pd.concat([condition_sbc, pd.DataFrame(pending_sbc)], ignore_index=True)
        condition_steps.to_csv(steps_path, index=False, compression='gzip')
        condition_actions.to_csv(actions_path, index=False, compression='gzip')
        if len(condition_sbc):
            condition_sbc.to_csv(sbc_path, index=False, compression='gzip')

    expected_steps = n_replicates * len(POLICIES) * (BUDGET_STEPS + 1)
    expected_actions = n_replicates * len(POLICIES) * BUDGET_STEPS
    if len(condition_steps) != expected_steps or len(condition_actions) != expected_actions:
        raise RuntimeError(
            f'Checkpoint row mismatch for {token}: steps {len(condition_steps)} != {expected_steps}; '
            f'actions {len(condition_actions)} != {expected_actions}'
        )
    if experiment_block == 'main' and scenario == 'model_matched' and np.isclose(strength, 1.0):
        expected_sbc = n_replicates * N_LATENT
        if len(condition_sbc) != expected_sbc:
            raise RuntimeError(f'SBC row mismatch: {len(condition_sbc)} != {expected_sbc}')

    all_step_frames.append(condition_steps)
    all_action_frames.append(condition_actions)
    if len(condition_sbc):
        all_sbc_frames.append(condition_sbc)
    completed_before += n_replicates

step_metrics = pd.concat(all_step_frames, ignore_index=True)
action_history = pd.concat(all_action_frames, ignore_index=True)
sbc_ranks = pd.concat(all_sbc_frames, ignore_index=True) if all_sbc_frames else pd.DataFrame()

print('\nALL PAIRED CAMPAIGNS COMPLETE')
print('  Step rows:', f'{len(step_metrics):,}')
print('  Action rows:', f'{len(action_history):,}')
print('  SBC rank rows:', f'{len(sbc_ranks):,}')


## 8. Paired inference, calibration checks, and claim gate


In [ ]:
METRICS = [
    'asset_uncertainty_fraction', 'equal_uncertainty_fraction',
    'asset_weighted_rmse', 'equal_weighted_rmse',
    'asset_weighted_interval_score_95', 'cellwise_50_coverage',
    'cellwise_80_coverage', 'cellwise_95_coverage', 'asset_weighted_95_coverage',
    'information_gain_nats', 'posterior_volume_fraction', 'cumulative_events',
]


def mean_normal_ci(values):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    half = float(1.96 * values.std(ddof=1) / np.sqrt(len(values))) if len(values) > 1 else 0.0
    return mean, mean - half, mean + half


def paired_bootstrap_ci(values, seed, draws=5000):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(values), size=(draws, len(values)))
    means = values[indices].mean(axis=1)
    return float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))


curve_rows = []
for keys, group in step_metrics.groupby(['experiment_block', 'scenario', 'strength', 'policy', 'step']):
    row = dict(zip(['experiment_block', 'scenario', 'strength', 'policy', 'step'], keys))
    for metric in METRICS:
        mean, low, high = mean_normal_ci(group[metric].to_numpy())
        row[f'{metric}_mean'] = mean
        row[f'{metric}_ci95_low'] = low
        row[f'{metric}_ci95_high'] = high
    row['n'] = len(group)
    curve_rows.append(row)
curve_summary = pd.DataFrame(curve_rows)

final_metrics = step_metrics.loc[step_metrics.step == BUDGET_STEPS].copy()
final_rows = []
for keys, group in final_metrics.groupby(['experiment_block', 'scenario', 'strength', 'policy']):
    row = dict(zip(['experiment_block', 'scenario', 'strength', 'policy'], keys))
    for metric in METRICS:
        mean, low, high = mean_normal_ci(group[metric].to_numpy())
        row[f'{metric}_mean'] = mean
        row[f'{metric}_ci95_low'] = low
        row[f'{metric}_ci95_high'] = high
    row['optimizer_failure_rate'] = float(1.0 - group.optimizer_success.mean())
    row['n'] = len(group)
    final_rows.append(row)
final_summary = pd.DataFrame(final_rows)

pairwise_rows = []
for condition_keys, condition in final_metrics.groupby(['experiment_block', 'scenario', 'strength']):
    experiment_block, scenario, strength = condition_keys
    proposed = condition.loc[condition.policy == 'asset_aopt'].set_index('replicate')
    for baseline in [policy for policy in POLICIES if policy != 'asset_aopt']:
        comparator = condition.loc[condition.policy == baseline].set_index('replicate')
        shared = proposed.index.intersection(comparator.index)
        baseline_uncertainty = comparator.loc[shared, 'asset_weighted_posterior_variance'].to_numpy()
        proposed_uncertainty = proposed.loc[shared, 'asset_weighted_posterior_variance'].to_numpy()
        uncertainty_improvement = 100.0 * (baseline_uncertainty - proposed_uncertainty) / np.maximum(baseline_uncertainty, 1e-12)
        baseline_rmse = comparator.loc[shared, 'asset_weighted_rmse'].to_numpy()
        proposed_rmse = proposed.loc[shared, 'asset_weighted_rmse'].to_numpy()
        rmse_improvement = 100.0 * (baseline_rmse - proposed_rmse) / np.maximum(baseline_rmse, 1e-12)
        baseline_score = comparator.loc[shared, 'asset_weighted_interval_score_95'].to_numpy()
        proposed_score = proposed.loc[shared, 'asset_weighted_interval_score_95'].to_numpy()
        interval_score_improvement = 100.0 * (baseline_score - proposed_score) / np.maximum(baseline_score, 1e-12)
        u_low, u_high = paired_bootstrap_ci(
            uncertainty_improvement, stable_seed('boot_unc', experiment_block, scenario, strength, baseline)
        )
        r_low, r_high = paired_bootstrap_ci(
            rmse_improvement, stable_seed('boot_rmse', experiment_block, scenario, strength, baseline)
        )
        s_low, s_high = paired_bootstrap_ci(
            interval_score_improvement, stable_seed('boot_score', experiment_block, scenario, strength, baseline)
        )
        pairwise_rows.append({
            'experiment_block': experiment_block, 'scenario': scenario, 'strength': strength,
            'baseline': baseline, 'paired_replicates': len(shared),
            'uncertainty_improvement_percent_mean': float(uncertainty_improvement.mean()),
            'uncertainty_improvement_ci95_low': u_low,
            'uncertainty_improvement_ci95_high': u_high,
            'uncertainty_win_rate': float(np.mean(uncertainty_improvement > 0)),
            'rmse_improvement_percent_mean': float(rmse_improvement.mean()),
            'rmse_improvement_ci95_low': r_low,
            'rmse_improvement_ci95_high': r_high,
            'rmse_win_rate': float(np.mean(rmse_improvement > 0)),
            'interval_score_improvement_percent_mean': float(interval_score_improvement.mean()),
            'interval_score_improvement_ci95_low': s_low,
            'interval_score_improvement_ci95_high': s_high,
        })
pairwise = pd.DataFrame(pairwise_rows)

                                                           
rank_bins = np.linspace(0, SBC_DRAWS + 1, 11)
rank_counts, _ = np.histogram(sbc_ranks['rank'].to_numpy(), bins=rank_bins)
rank_chi2, rank_pvalue = chisquare(rank_counts)
sbc_by_shell_rows = []
for altitude, group in sbc_ranks.groupby('altitude_center_km'):
    counts, _ = np.histogram(group['rank'].to_numpy(), bins=rank_bins)
    statistic, pvalue = chisquare(counts)
    sbc_by_shell_rows.append({
        'altitude_center_km': altitude, 'chi_square': float(statistic),
        'p_value': float(pvalue), 'rank_bin_counts': json.dumps(counts.tolist()),
    })
sbc_by_shell = pd.DataFrame(sbc_by_shell_rows)

primary_pairwise = pairwise.loc[
    (pairwise.experiment_block == 'main')
    & (pairwise.scenario == 'model_matched')
    & np.isclose(pairwise.strength, 1.0)
].copy()
primary_proposed = final_summary.loc[
    (final_summary.experiment_block == 'main')
    & (final_summary.scenario == 'model_matched')
    & np.isclose(final_summary.strength, 1.0)
    & (final_summary.policy == 'asset_aopt')
].iloc[0]

core_baselines = {'best_fixed', 'random', 'ordem_flux_fixed'}
core_pairs = primary_pairwise.loc[primary_pairwise.baseline.isin(core_baselines)]
claim_checks = {
    'beats_best_fixed_random_and_ordem_flux_on_primary_endpoint': bool(
        len(core_pairs) == 3 and (core_pairs.uncertainty_improvement_ci95_low > 0).all()
    ),
    'model_matched_95_coverage_between_90_and_98_percent': bool(
        0.90 <= primary_proposed.cellwise_95_coverage_mean <= 0.98
    ),
    'optimizer_failure_rate_below_1_percent': bool(primary_proposed.optimizer_failure_rate < 0.01),
    'fine_grid_loo_total_flux_95th_below_1_percent': bool(loo.total_flux_relative_error.quantile(0.95) < 0.01),
    'fine_grid_loo_shape_l1_95th_below_5_percent': bool(loo.normalized_shape_l1_error.quantile(0.95) < 0.05),
}
claim_supported = all(claim_checks.values())
claim_status = {
    'status': 'supported_in_predeclared_simulation_suite' if claim_supported else 'not_yet_supported_by_every_predeclared_gate',
    'central_claim': (
        'Under equal simulated radar time, active-satellite-exposure-weighted A-optimal pointing '
        'reduces posterior population uncertainty relative to strong fixed, random, and ORDEM-flux-only schedules.'
    ),
    'checks': claim_checks,
    'primary_pairwise': core_pairs.to_dict(orient='records'),
    'model_matched_proposed_coverage': float(primary_proposed.cellwise_95_coverage_mean),
    'aggregate_sbc_rank_chi_square': float(rank_chi2),
    'aggregate_sbc_rank_p_value_descriptive_only': float(rank_pvalue),
    'warning': (
        'This claim concerns simulated future tasking calibrated by real radar counts. '
        'It does not claim historical adaptive steering, individual trajectories, or collision warnings.'
    ),
}

display(final_summary.sort_values(['experiment_block', 'scenario', 'strength', 'policy']))
print('\nPrimary paired comparisons:')
display(primary_pairwise)
print('\nPredeclared claim status:', claim_status['status'])
print('Claim gates:', claim_checks)


## 9. Publication figures


In [ ]:
POLICY_LABELS = {
    'best_fixed': 'Best fixed',
    'random': 'Random',
    'ordem_flux_fixed': 'ORDEM-flux fixed',
    'unweighted_aopt': 'Unweighted A-opt',
    'eig': 'EIG',
    'asset_aopt': 'Satellite-weighted A-opt',
}
POLICY_COLORS = {
    'best_fixed': '#595959',
    'random': '#4C78A8',
    'ordem_flux_fixed': '#F58518',
    'unweighted_aopt': '#B279A2',
    'eig': '#54A24B',
    'asset_aopt': '#E45756',
}

plt.rcParams.update({'font.size': 10, 'figure.dpi': 130})
primary_curve = curve_summary.loc[
    (curve_summary.experiment_block == 'main')
    & (curve_summary.scenario == 'model_matched')
    & np.isclose(curve_summary.strength, 1.0)
].copy()

fig, axes = plt.subplots(2, 2, figsize=(13.8, 10.2), constrained_layout=True)
for policy in POLICIES:
    data = primary_curve.loc[primary_curve.policy == policy].sort_values('step')
    axes[0, 0].plot(
        data.step, data.asset_uncertainty_fraction_mean,
        color=POLICY_COLORS[policy], lw=2, label=POLICY_LABELS[policy],
    )
    axes[0, 0].fill_between(
        data.step, data.asset_uncertainty_fraction_ci95_low, data.asset_uncertainty_fraction_ci95_high,
        color=POLICY_COLORS[policy], alpha=0.10,
    )
axes[0, 0].set(
    title='A. Active-satellite-weighted uncertainty',
    xlabel='One-hour observations', ylabel='Posterior variance / prior variance',
)
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend(frameon=False, fontsize=8, ncol=2)

for policy in POLICIES:
    data = primary_curve.loc[primary_curve.policy == policy].sort_values('step')
    axes[0, 1].plot(
        data.step, data.asset_weighted_rmse_mean,
        color=POLICY_COLORS[policy], lw=2, label=POLICY_LABELS[policy],
    )
    axes[0, 1].fill_between(
        data.step, data.asset_weighted_rmse_ci95_low, data.asset_weighted_rmse_ci95_high,
        color=POLICY_COLORS[policy], alpha=0.10,
    )
axes[0, 1].set(
    title='B. Active-satellite-weighted reconstruction error',
    xlabel='One-hour observations', ylabel='RMSE of ORDEM multiplier',
)
axes[0, 1].grid(alpha=0.25)

scenario_order = list(MAIN_SCENARIOS)
baseline_order = ['best_fixed', 'random', 'ordem_flux_fixed', 'unweighted_aopt', 'eig']
main_pairs = pairwise.loc[
    (pairwise.experiment_block == 'main') & np.isclose(pairwise.strength, 1.0)
].copy()
x_positions = np.arange(len(scenario_order))
offsets = np.linspace(-0.28, 0.28, len(baseline_order))
for offset, baseline in zip(offsets, baseline_order):
    subset = main_pairs.loc[main_pairs.baseline == baseline].set_index('scenario').loc[scenario_order]
    axes[1, 0].errorbar(
        x_positions + offset,
        subset.uncertainty_improvement_percent_mean,
        yerr=[
            subset.uncertainty_improvement_percent_mean - subset.uncertainty_improvement_ci95_low,
            subset.uncertainty_improvement_ci95_high - subset.uncertainty_improvement_percent_mean,
        ],
        fmt='o', capsize=3, color=POLICY_COLORS[baseline], label=f"vs {POLICY_LABELS[baseline]}",
    )
axes[1, 0].axhline(0, color='black', lw=1)
axes[1, 0].set_xticks(x_positions, ['Matched', 'Local anomaly', 'Shell mixing', 'ORDEM response\nperturbation'])
axes[1, 0].set(
    title='C. Proposed-policy uncertainty advantage',
    ylabel='Paired improvement (%)',
)
axes[1, 0].grid(axis='y', alpha=0.25)
axes[1, 0].legend(frameon=False, fontsize=7, ncol=2)

selection_primary = action_history.loc[
    (action_history.experiment_block == 'main')
    & (action_history.scenario == 'model_matched')
    & np.isclose(action_history.strength, 1.0)
]
for policy in POLICIES:
    counts = selection_primary.loc[selection_primary.policy == policy].elevation_deg.value_counts().reindex(ELEVATIONS, fill_value=0)
    probability = counts.to_numpy(dtype=float) / max(counts.sum(), 1)
    axes[1, 1].plot(ELEVATIONS, probability, color=POLICY_COLORS[policy], lw=1.8, label=POLICY_LABELS[policy])
axes[1, 1].set(
    title='D. Elevations selected under the same test worlds',
    xlabel='Elevation (deg)', ylabel='Fraction of observations',
)
axes[1, 1].grid(alpha=0.25)

fig.suptitle('Final fine-grid adaptive radar-scheduling experiment', fontsize=15)
fig.savefig(OUTPUT_DIR / 'final_adaptive_policy_main_2x2.png', dpi=240, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'final_adaptive_policy_main_2x2.pdf', bbox_inches='tight')
plt.show()


fig, axes = plt.subplots(2, 2, figsize=(13.8, 10.2), constrained_layout=True)
image = axes[0, 0].imshow(
    H.T / np.maximum(H.sum(axis=1), 1e-12), origin='lower', aspect='auto', cmap='magma',
    extent=[ELEVATIONS.min() - 0.25, ELEVATIONS.max() + 0.25, ALTITUDE_EDGES.min(), ALTITUDE_EDGES.max()],
)
axes[0, 0].set(
    title='A. Exact ORDEM response shapes', xlabel='Elevation (deg)', ylabel='Altitude shell (km)',
)
fig.colorbar(image, ax=axes[0, 0], label='Fraction of response')

axes[0, 1].plot(loo.elevation_deg, 100 * loo.total_flux_relative_error, label='Total flux', lw=2)
axes[0, 1].plot(loo.elevation_deg, 100 * loo.normalized_shape_l1_error, label='Normalized shape L1', lw=2)
axes[0, 1].axhline(1.0, color='gray', lw=1, linestyle=':', label='1% reference')
axes[0, 1].set(
    title='B. Leave-one-elevation-out interpolation audit',
    xlabel='Held-out elevation (deg)', ylabel='Relative error (%)',
)
axes[0, 1].grid(alpha=0.25)
axes[0, 1].legend(frameon=False)

axes[1, 0].bar(np.arange(10), rank_counts, color='#4C78A8', width=0.82)
axes[1, 0].axhline(rank_counts.mean(), color='black', lw=1, linestyle='--', label='Uniform expectation')
axes[1, 0].set(
    title=f'C. SBC ranks, proposed policy (descriptive p={rank_pvalue:.3f})',
    xlabel='Posterior-rank decile', ylabel='Count',
)
axes[1, 0].set_xticks(np.arange(10), [str(i + 1) for i in range(10)])
axes[1, 0].legend(frameon=False)

hours = np.arange(24)
axes[1, 1].plot(hours, HOURLY_COUNTS, marker='o', color='black', lw=1.5, label='Held-out real count')
axes[1, 1].fill_between(
    hours, count_cv.nb_95_low.to_numpy(), count_cv.nb_95_high.to_numpy(),
    color='#4C78A8', alpha=0.18, label='Fold-specific NB 95% interval',
)
axes[1, 1].plot(hours, count_cv.training_mean, color='#4C78A8', lw=2, label='Training-fold mean')
axes[1, 1].set(
    title='D. Blocked prediction of untouched real hours',
    xlabel='Hour in 24-hour campaign', ylabel='Accepted events',
)
axes[1, 1].grid(alpha=0.25)
axes[1, 1].legend(frameon=False, fontsize=8)

fig.suptitle('Validation and robustness diagnostics', fontsize=15)
fig.savefig(OUTPUT_DIR / 'final_validation_diagnostics_2x2.png', dpi=240, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'final_validation_diagnostics_2x2.pdf', bbox_inches='tight')
plt.show()


## 10. Save the reproducible research package and enforce quality checks


In [ ]:
FINE_GRID_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / 'b982_fine_elevation_49'
FINE_GRID_DIR.mkdir(parents=True, exist_ok=True)
fine_grid_response = pd.DataFrame(H, columns=[f'shell_{center:g}_km' for center in ALTITUDE_CENTERS])
fine_grid_response.insert(0, 'elevation_deg', ELEVATIONS)
fine_grid_response.insert(0, 'azimuth_deg', 90.0)
fine_grid_response.insert(0, 'action_id', ACTION_IDS)
fine_grid_response.to_csv(FINE_GRID_DIR / 'ordem_fine_elevation_response_matrix_49.csv', index=False)
inventory.to_csv(FINE_GRID_DIR / 'ordem_refinement_inventory_40.csv', index=False)
uncertainty_elements.to_csv(FINE_GRID_DIR / 'ordem_refinement_uncertainty_elements.csv.gz', index=False, compression='gzip')
response_uncertainty.to_csv(FINE_GRID_DIR / 'ordem_refinement_action_uncertainty.csv', index=False)
loo.to_csv(FINE_GRID_DIR / 'ordem_fine_grid_leave_one_out_audit.csv', index=False)
(FINE_GRID_DIR / 'ordem_fine_grid_action_audit.json').write_text(json.dumps(action_audit, indent=2))
count_cv.to_csv(OUTPUT_DIR / 'real_hourly_count_blocked_cv.csv', index=False)
(OUTPUT_DIR / 'real_hourly_count_blocked_cv_summary.json').write_text(json.dumps(count_cv_summary, indent=2))
fixed_design.to_csv(OUTPUT_DIR / 'strong_fixed_elevation_design.csv', index=False)
step_metrics.to_csv(OUTPUT_DIR / 'final_policy_step_metrics.csv.gz', index=False, compression='gzip')
action_history.to_csv(OUTPUT_DIR / 'final_policy_action_history.csv.gz', index=False, compression='gzip')
curve_summary.to_csv(OUTPUT_DIR / 'final_policy_curve_summary.csv', index=False)
final_summary.to_csv(OUTPUT_DIR / 'final_policy_summary.csv', index=False)
pairwise.to_csv(OUTPUT_DIR / 'final_policy_paired_comparisons.csv', index=False)
sbc_ranks.to_csv(OUTPUT_DIR / 'simulation_based_calibration_ranks.csv.gz', index=False, compression='gzip')
sbc_by_shell.to_csv(OUTPUT_DIR / 'simulation_based_calibration_by_shell.csv', index=False)
(OUTPUT_DIR / 'final_claim_status.json').write_text(json.dumps(claim_status, indent=2))
expected_condition_groups = len(experiment_conditions)
quality_checks = {'refinement_zip_checksum_verified': refinement_hash == EXPECTED_REFINEMENT_SHA256, 'forty_new_jobs_complete': len(inventory) == 40 and bool(inventory.complete.all()), 'forty_new_elevations_unique_and_exact': actual_new_elevations == expected_new_elevations, 'forty_jobs_have_range_and_population_uncertainty': len(response_uncertainty) == 40, 'forty_nine_exact_primary_actions': N_ACTIONS == 49 and np.allclose(ELEVATIONS, expected_all), 'no_interpolation_in_primary_operator': H.shape == (49, 10), 'all_predeclared_conditions_present': step_metrics.groupby(['experiment_block', 'scenario', 'strength']).ngroups == expected_condition_groups, 'all_policies_present': set(step_metrics.policy.unique()) == set(POLICIES), 'all_budgets_equal': bool((action_history.groupby(['experiment_block', 'scenario', 'strength', 'replicate', 'policy']).size() == BUDGET_STEPS).all()), 'best_fixed_policy_is_fixed': bool((action_history.loc[action_history.policy == 'best_fixed', 'action_index'] == BEST_FIXED_ACTION).all()), 'ordem_flux_policy_is_fixed': bool((action_history.loc[action_history.policy == 'ordem_flux_fixed', 'action_index'] == ORDEM_FLUX_ACTION).all()), 'finite_primary_metrics': bool(np.isfinite(step_metrics[['asset_uncertainty_fraction', 'equal_uncertainty_fraction', 'asset_weighted_rmse', 'asset_weighted_interval_score_95', 'cellwise_95_coverage', 'information_gain_nats']].to_numpy()).all()), 'overall_optimizer_failure_rate_below_1_percent': float(1.0 - step_metrics.optimizer_success.mean()) < 0.01, 'sbc_rows_complete': len(sbc_ranks) == N_MAIN_REPLICATES * N_LATENT, 'held_out_real_hours_complete': len(count_cv) == 24 and count_cv.hour_key.nunique() == 24}
failed_quality = [name for name, passed in quality_checks.items() if not passed]
if failed_quality:
    raise RuntimeError(f'Final quality checks failed: {failed_quality}')
manifest = {'created_utc': datetime.now(timezone.utc).isoformat(), 'status': 'complete', 'config_hash': config_hash, 'configuration': experiment_config, 'ordem_action_audit': action_audit, 'real_count_validation': count_cv_summary, 'simulation_based_calibration': {'rank_rows': int(len(sbc_ranks)), 'rank_decile_counts': rank_counts.tolist(), 'aggregate_chi_square_descriptive': float(rank_chi2), 'aggregate_p_value_descriptive': float(rank_pvalue), 'caution': 'Ranks are correlated across altitude cells; coverage is the primary calibration diagnostic.'}, 'quality_checks': quality_checks, 'claim_status': claim_status, 'real_radar_calibration': {'consolidated_events': 155, 'hours': 24, 'tier_a_strict': 46, 'tier_b_empirical': 3, 'unresolved': 106, 'unresolved_interpretation': 'Unresolved is not asserted to mean uncataloged debris.'}, 'limitations': ['The 2018 EISCAT observations used for calibration were not adaptively steered.', 'The primary action set is one azimuth and 49 elevation samples; it does not resolve full-sky direction or inclination.', 'ORDEM is a prior and response model, not independent ground truth.', 'The real-data cross-validation tests hourly counts, not whether each unresolved event is physical debris.', 'All active satellites receive equal object weight; this is not a critical-service importance score.', 'The method estimates a statistical altitude-shell population and does not recover individual trajectories or collision probabilities.', 'Independent Svalbard processing and expected-catalog-passage completeness remain separate external detector/sensor validation tasks.']}
(OUTPUT_DIR / 'final_run_manifest.json').write_text(json.dumps(manifest, indent=2))
(OUTPUT_DIR / '_SUCCESS.json').write_text(json.dumps({'status': 'complete', 'created_utc': manifest['created_utc'], 'config_hash': config_hash, 'claim_status': claim_status['status'], 'quality_checks': quality_checks}, indent=2))
readme = f"research final fine-grid Bayesian adaptive-observation experiment\n===========================================================\n\nCompleted input\n---------------\n- ORDEM job b9824543-c020-42b7-b747-61b298670ce4: 40/40 complete, SHA256 verified.\n- Exact primary action grid: 49 elevations, 65-89 degrees in 0.5-degree steps, AZ=90 degrees.\n- Real radar calibration: 155 accepted events across 24 hours.\n- Real active-satellite exposure: equal weight per active object.\n\nPrimary policy\n--------------\nActive-satellite-exposure-weighted Bayesian A-optimal design.\n\nBaselines\n---------\n- Strong best fixed elevation selected before test worlds.\n- Random elevation.\n- ORDEM-flux-only fixed elevation.\n- Unweighted A-optimal adaptive design.\n- Expected information gain.\n\nValidation layers\n-----------------\n- ORDEM fine-grid leave-one-elevation-out audit.\n- Six-fold blocked prediction of untouched real hourly counts.\n- Simulation-based calibration under model-matched truth.\n- Local anomaly, neighbor-shell mixing, and ORDEM-response perturbation stress tests.\n- Paired bootstrap confidence intervals using common hidden worlds and observation streams.\n\nClaim status\n------------\n{claim_status['status']}\n\nThis result concerns simulated future tasking calibrated by real EISCAT measurements. It is not historical adaptive steering, individual-object tracking, or collision warning.\n"
(OUTPUT_DIR / 'README_final_experiment.txt').write_text(readme)
print('\n' + '=' * 84)
print('SUCCESS — FINAL FINE-GRID BAYESIAN EXPERIMENT COMPLETE')
print('=' * 84)
print('Config hash:', config_hash)
print('Output folder:', OUTPUT_DIR)
print('Claim status:', claim_status['status'])
print('Best fixed elevation:', ELEVATIONS[BEST_FIXED_ACTION])
print('ORDEM-flux-only elevation:', ELEVATIONS[ORDEM_FLUX_ACTION])
print('\nPrimary paired comparisons:')
display(primary_pairwise[['baseline', 'uncertainty_improvement_percent_mean', 'uncertainty_improvement_ci95_low', 'uncertainty_improvement_ci95_high', 'uncertainty_win_rate', 'rmse_improvement_percent_mean', 'interval_score_improvement_percent_mean']])
print('\nNext external validation still required for the manuscript:')
print('  1. Expected catalog passages -> detected/missed completeness audit.')
print('  2. Frozen small Svalbard sample, if the raw archive is obtainable.')


## Outputs
